# Notebook 2 (Fase 1) — Clasificador SVC → MongoDB
### Ernesto Investing AI · iDeSo · UNMSM — Semana 13

**Objetivo:** Leer datos de MongoDB, entrenar un clasificador SVC (Support Vector Classifier) con `GridSearchCV` para predecir señales BUY/SELL, y guardar las predicciones y métricas de vuelta en MongoDB.

**Prerequisito:** Haber ejecutado el Notebook 1 (colección `precios_ohlcv` poblada).

**Salida:** Colecciones `predicciones` y `metricas_modelos` pobladas para los 5 tickers.

**Pipeline:** MongoDB (`precios_ohlcv`) → features + target → SVC + GridSearchCV → métricas → MongoDB (`predicciones`, `metricas_modelos`)

In [ ]:
# Paso 1 — Instalar librerias necesarias
!pip install "pymongo[srv]" scikit-learn --quiet

print("Librerias instaladas correctamente")

In [ ]:
# Paso 2 — Conectar a MongoDB Atlas
# La contrasena NUNCA se escribe en texto plano en el notebook: se pide de forma
# oculta con getpass y se inserta en la URI de conexion en tiempo de ejecucion.
# Alternativa con Secrets de Colab:
# from google.colab import userdata; MONGO_URI = userdata.get('MONGO_URI')
from getpass import getpass
from pymongo import MongoClient
from pymongo.server_api import ServerApi
import pandas as pd
import numpy as np

DB_USER = "gdev03_db_user"
DB_PASSWORD = getpass("Password de MongoDB Atlas: ")

MONGO_URI = (
    f"mongodb+srv://{DB_USER}:{DB_PASSWORD}"
    "@cluster0.sxjck8h.mongodb.net/?appName=Cluster0"
)

client = MongoClient(MONGO_URI, server_api=ServerApi('1'))

# Verificar la conexion con un ping, manejando errores de forma explicita
try:
    client.admin.command("ping")
    print("Conectado a MongoDB Atlas correctamente")
except Exception as e:
    raise Exception("Ocurrio el siguiente error al conectar: ", e)

db = client["ernesto_investing_ai"]

In [ ]:
# Paso 3 — Cargar datos de MongoDB (coleccion precios_ohlcv del Notebook 1)
def cargar_datos(ticker):
    """Lee de MongoDB los precios OHLCV + indicadores de un ticker, ordenados por fecha."""
    docs = list(db["precios_ohlcv"].find({"ticker": ticker}).sort("fecha", 1))
    return pd.DataFrame(docs)

df = cargar_datos("BVN")
print(f"Cargados {len(df)} dias de BVN desde MongoDB")
print(df[["fecha", "close", "sma_20", "rsi_14"]].tail())

In [ ]:
# Paso 4 — Ingenieria de features y variable objetivo
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

def preparar_features(df):
    """Construye las features y el target binario (1=BUY, 0=SELL).

    IMPORTANTE (sin data leakage): el target se calcula con el precio de
    CIERRE DEL DIA SIGUIENTE (shift(-1)), es decir, informacion futura respecto
    a la fila actual. Por eso el ultimo registro se descarta (no tiene target
    conocido) y por eso el entrenamiento en el Paso 5 usa una particion
    TEMPORAL (no aleatoria): el bloque de test siempre queda despues del
    bloque de train en el tiempo, nunca se mezclan fechas futuras en el train.
    """
    df = df.copy()
    df["retorno"] = df["close"].pct_change()

    # Target: 1 (BUY) si el dia siguiente sube, 0 (SELL) si baja
    df["target"] = (df["close"].shift(-1) > df["close"]).astype(int)

    features = ["close", "sma_20", "sma_50", "ema_12", "ema_26", "rsi_14", "retorno"]

    # Se descartan filas con NaN (inicio de las medias moviles y ultima fila sin target)
    df = df.dropna(subset=features + ["target"])

    X = df[features].values
    y = df["target"].values
    return X, y

X, y = preparar_features(df)
print(f"Features: {X.shape[0]} muestras, {X.shape[1]} caracteristicas")
print(f"Clases: BUY={sum(y)}, SELL={len(y) - sum(y)}")

In [ ]:
# Paso 5 — Entrenar SVC con GridSearchCV (particion temporal 80/20, sin data leakage)
def entrenar_svc(X, y):
    """Entrena un SVC optimizado con GridSearchCV y devuelve modelo, scaler y metricas."""
    n = len(X)
    corte = int(n * 0.8)

    # Particion TEMPORAL: los primeros 80% de fechas para train, el 20% final para test.
    # Nunca se usan datos del futuro (test) para entrenar (train).
    X_train, X_test = X[:corte], X[corte:]
    y_train, y_test = y[:corte], y[corte:]

    # El scaler se ajusta SOLO con train, y se aplica (transform) a test,
    # para no filtrar informacion estadistica del futuro hacia el pasado.
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    param_grid = {
        "kernel": ["linear", "rbf"],
        "C": [0.1, 1, 10],
        "gamma": ["scale", "auto"]
    }

    grid = GridSearchCV(SVC(probability=True), param_grid, cv=3, scoring="f1")
    grid.fit(X_train_s, y_train)

    mejor = grid.best_estimator_
    y_pred = mejor.predict(X_test_s)

    metricas = {
        "accuracy": round(accuracy_score(y_test, y_pred), 4),
        "precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "mejor_kernel": grid.best_params_["kernel"],
        "mejor_C": grid.best_params_["C"],
        "mejor_gamma": str(grid.best_params_["gamma"])
    }
    cm = confusion_matrix(y_test, y_pred).tolist()

    return mejor, scaler, metricas, cm

modelo, scaler, metricas, cm = entrenar_svc(X, y)
acc = metricas["accuracy"]
f1v = metricas["f1"]
kern = metricas["mejor_kernel"]
cval = metricas["mejor_C"]

print("SVC entrenado para BVN")
print(f"  Accuracy: {acc:.2%}")
print(f"  F1-Score: {f1v:.2%}")
print(f"  Mejor kernel: {kern}, C={cval}")
print(f"  Matriz de confusion: {cm}")

In [ ]:
# Paso 6 — Procesar un ticker completo: entrenar, predecir y guardar en MongoDB
from datetime import datetime

def procesar_ticker(ticker):
    """Pipeline completo para un ticker: carga -> features -> entrena -> predice -> guarda."""
    df = cargar_datos(ticker)

    if len(df) < 100:
        print(f"  [{ticker}] Datos insuficientes, se omite.")
        return

    X, y = preparar_features(df)
    modelo, scaler, metricas, cm = entrenar_svc(X, y)

    # Prediccion de la senal actual usando la ultima fila disponible con features completas
    X_norm = scaler.transform(X)
    pred = int(modelo.predict(X_norm[-1:])[0])
    prob = float(modelo.predict_proba(X_norm[-1:])[0].max())
    senal = "BUY" if pred == 1 else "SELL"

    # Guardar prediccion (se borra la anterior del mismo ticker/modelo para evitar duplicados)
    db["predicciones"].delete_many({"ticker": ticker, "modelo": "SVC"})
    db["predicciones"].insert_one({
        "ticker": ticker,
        "modelo": "SVC",
        "senal": senal,
        "confianza": round(prob, 4),
        "fecha_prediccion": datetime.now().strftime("%Y-%m-%d"),
        "created_at": datetime.now()
    })

    # Guardar metricas del entrenamiento
    db["metricas_modelos"].delete_many({"ticker": ticker, "modelo": "SVC"})
    db["metricas_modelos"].insert_one({
        "ticker": ticker,
        "modelo": "SVC",
        **metricas,
        "matriz_confusion": cm,
        "fecha_entrenamiento": datetime.now()
    })

    acc = metricas["accuracy"]
    print(f"  [{ticker}] senal={senal} ({prob:.0%}) | accuracy={acc:.0%} | guardado")

print("Funcion procesar_ticker lista")

In [ ]:
# Paso 7 — Procesar los 5 tickers del proyecto
TICKERS = ["FSM", "VOLCABC1.LM", "ABX.TO", "BVN", "BHP"]

print("Entrenando SVC para los 5 tickers...")
print()

for ticker in TICKERS:
    try:
        procesar_ticker(ticker)
    except Exception as e:
        print(f"  [{ticker}] Error: {e}")

print()
print("Procesamiento completo!")

In [ ]:
# Paso 8 — Celda de verificacion final: resultados guardados en MongoDB
print("Predicciones SVC en MongoDB:")
print()

tickers_con_prediccion = set()
for doc in db["predicciones"].find({"modelo": "SVC"}):
    t = doc["ticker"]
    s = doc["senal"]
    c = doc["confianza"]
    tickers_con_prediccion.add(t)
    print(f"  {t}: {s} (confianza {c:.0%})")

print()
print("Metricas SVC en MongoDB:")
print()

tickers_con_metricas = set()
for doc in db["metricas_modelos"].find({"modelo": "SVC"}):
    t = doc["ticker"]
    a = doc["accuracy"]
    f = doc["f1"]
    tickers_con_metricas.add(t)
    print(f"  {t}: accuracy={a:.0%}, f1={f:.0%}")

print()
faltantes = set(TICKERS) - tickers_con_prediccion - tickers_con_metricas
faltantes |= (set(TICKERS) - tickers_con_prediccion) | (set(TICKERS) - tickers_con_metricas)
if not faltantes:
    print("Verificacion exitosa: los 5 tickers tienen prediccion y metricas en MongoDB.")
else:
    print(f"Atencion: faltan resultados para: {sorted(faltantes)}")

## Resultado

Las colecciones `predicciones` y `metricas_modelos` contienen resultados reales del clasificador SVC (señal BUY/SELL, confianza, accuracy, precision, recall, F1 y matriz de confusión) para los 5 tickers.

La **API FastAPI (Fase 2)** leerá estas colecciones y las servirá al frontend.

**Pipeline:** MongoDB (`precios_ohlcv`) → SVC + GridSearchCV → **predicciones y métricas en MongoDB** ✓